In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("atharvjairath/empathetic-dialogues-facebook-ai")

print("Path to dataset files:", path)

100%|██████████| 3.26M/3.26M [00:00<00:00, 5.18MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/atharvjairath/empathetic-dialogues-facebook-ai/versions/1


In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/atharvjairath/empathetic-dialogues-facebook-ai/versions/1/emotion-emotion_69k.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
import os
df = pd.read_csv(os.path.join(path, "emotion-emotion_69k.csv"))

In [ ]:
print(df.shape)

print(df.columns)

df.head()

(64636, 7)
Index(['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')


,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


In [ ]:
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 5', 'Unnamed: 6'])

In [ ]:
df.head()

df.isna().sum()

,0
Situation,0
emotion,4
empathetic_dialogues,0
labels,0


In [2]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ", text) #lines
  text=re.sub(r"\s+"," ",text) #remove spaces
  text=re.sub(r"<.*?>"," ",text) #html tags


  text = text.strip().lower()

  return text

In [ ]:
# df["Situation"] = df["Situation"].apply(clean_data)
# df["emotion"] = df["emotion"].fillna('').apply(clean_data)
# df["empathetic_dialogues"] = df["empathetic_dialogues"].apply(clean_data)
# df["labels"] = df["labels"].apply(clean_data)

for col in ["Situation", "emotion", "empathetic_dialogues", "labels"]:
    df[col] = df[col].fillna("").astype(str).apply(clean_data)

In [ ]:
df.head()

,Situation,emotion,empathetic_dialogues,labels
0,i remember going to the fireworks with my best...,sentimental,customer :i remember going to see the firework...,"was this a friend you were in love with, or ju..."
1,i remember going to the fireworks with my best...,sentimental,customer :this was a best friend. i miss her. ...,where has she gone?
2,i remember going to the fireworks with my best...,sentimental,customer :we no longer talk. agent :,oh was this something that happened because of...
3,i remember going to the fireworks with my best...,sentimental,customer :was this a friend you were in love w...,this was a best friend. i miss her.
4,i remember going to the fireworks with my best...,sentimental,customer :where has she gone? agent :,we no longer talk.


In [ ]:
train,valid = train_test_split(
    df,
    test_size=0.3,
    random_state = 42,
    shuffle=True
)

# valid,test = train_test_split(
#     temp,
#     test_size=0.5,
#     random_state=42,
#     shuffle=True
# )

In [ ]:
!pip install "transformers"
!pip install "transformers[torch]"

In [3]:
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

# Tokenize

In [4]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

In [ ]:
train.head()

,Situation,emotion,empathetic_dialogues,labels
10517,when my car insurance got turned off bc someon...,furious,customer :my wife forgot to pay the car insura...,"that's unfortunate, did you end up with a ticket?"
13602,i remember when a hamburger was just a dollar....,nostalgic,customer :inflation sucks agent :,"yeah, those were the good old days."
45959,i can't wait till the weekend! i'm going to a ...,anticipating,customer :i'm thinking of a waterpark we have ...,why don't you take them to six flags. it is mo...
31167,i'm afraid to leave my current job (which i ha...,afraid,customer :yes. and the tricky part is that you...,at least it sounds like you're giving it a lot...
53595,dog greeting me at the door whenever i get home,caring,customer :i have a little havanese named yoko....,that's an awesome name. i have two feists. if ...


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train, preserve_index=False)
valid_ds = Dataset.from_pandas(valid, preserve_index=False)

In [5]:
import torch

def tokenize(data):

  prompt = (
        f"You are an empathetic mentor.\n\n"
        f"Emotion: {data['emotion']}\n\n"
        f"Situation: {data['Situation']}\n\n"
        f"{data['empathetic_dialogues']}"
    )

  inputs = tokenizer(prompt, padding="max_length", max_length=128, truncation=True, return_tensors="pt")


  targets = tokenizer(data["labels"], padding="max_length", max_length=128, truncation=True, return_tensors="pt")

  # Replace padding token id (0) with -100 for loss calculation
  labels_with_ignore_index = targets["input_ids"].clone()
  labels_with_ignore_index[labels_with_ignore_index == tokenizer.pad_token_id] = -100

  # Remove the batch dimension added by return_tensors="pt" for single examples
  return {
      "input_ids": inputs["input_ids"].squeeze(0),
      "attention_mask": inputs["attention_mask"].squeeze(0),
      "labels": labels_with_ignore_index.squeeze(0)
  }

In [ ]:
train_ds = train_ds.map(
    tokenize,
    remove_columns=train_ds.column_names
)

valid_ds = valid_ds.map(
    tokenize,
    remove_columns=valid_ds.column_names
)

Map:   0%|          | 0/45245 [00:00<?, ? examples/s]

Map:   0%|          | 0/19391 [00:00<?, ? examples/s]

In [ ]:
print(type(train))
print(type(train_ds))
train_ds[0]


<class 'pandas.core.frame.DataFrame'>
<class 'datasets.arrow_dataset.Dataset'>


{'labels': [24,
  31,
  7,
  20343,
  6,
  410,
  25,
  414,
  95,
  28,
  3,
  9,
  4142,
  58,
  1,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  

# model making

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs =3,
    weight_decay = 0.01,

    learning_rate=3e-5,

    per_device_train_batch_size = 2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size = 2,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    load_best_model_at_end=True,
    logging_steps=100,
    save_total_limit=2

)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = valid_ds
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,10.445460,2.408826


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,10.445460,2.408826
2,9.971659,2.381134
3,9.883575,2.373408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=16968, training_loss=10.117101537119044, metrics={'train_runtime': 12739.6795, 'train_samples_per_second': 10.655, 'train_steps_per_second': 1.332, 'total_flos': 2.323638480863232e+16, 'train_loss': 10.117101537119044, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
9.883575,2.373408,3


{'eval_loss': 2.3734078407287598}

In [ ]:
trainer.save_model("./best_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!zip -r results.zip ./best_model

  adding: best_model/ (stored 0%)
  adding: best_model/generation_config.json (deflated 29%)
  adding: best_model/model.safetensors (deflated 7%)
  adding: best_model/config.json (deflated 63%)
  adding: best_model/training_args.bin (deflated 53%)


In [ ]:
from google.colab import files
files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Once you have downloaded the `results.zip` file, you can load the trained model and see how it responds to a personalized message.

In [1]:
# Load the trained model
from transformers import AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSeq2SeqLM.from_pretrained("./best_model")
model.to(device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

You can use the `generate_response` function to get a reply from the model. Try changing the `emotion`, `situation`, and `dialogue` to test different scenarios.

In [ ]:
def generate_response(emotion, situation, dialogue):
    prompt = f"""
You are an empathetic mentor.

Emotion: {emotion}

Situation: {situation}

Student says:
{dialogue}

Give a supportive response appropriate for a student who feels {emotion}.

Response:
"""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
    input_ids,
    max_new_tokens=80,
    min_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.92,
    repetition_penalty=1.2
)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example of a personalized message
my_emotion = "sad"
my_situation = "I failed my exam."
my_dialogue = "i was not fully prepared and feeling a sense of giving up"

response = generate_response(my_emotion, my_situation, my_dialogue)
print(f"Model's reply: {response}")

Model's reply: i know what you mean. i am sure you can work through it. thank you! i hope you are doing well. i am sure you will feel better soon.


In [42]:
emotion_templates = {
    "sad": """
You are an empathetic mentor.

The student is feeling sad.

Your response should:
- Acknowledge their disappointment.
- Reassure them that setbacks are temporary.
- Offer one practical next step.
- End with encouragement.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "anxious": """
You are an empathetic mentor.

The student is feeling anxious.

Your response should:
- Validate their worries.
- Help them focus on what they can control.
- Suggest a calming or planning strategy.
- End with reassurance.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "stressed": """
You are an empathetic mentor.

The student is feeling stressed.

Your response should:
- Acknowledge the pressure they are facing.
- Encourage self-care and balance.
- Suggest one practical way to reduce stress.
- End positively.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "lonely": """
You are an empathetic mentor.

The student is feeling lonely.

Your response should:
- Acknowledge their feelings.
- Remind them that many students experience loneliness.
- Suggest healthy ways to connect with others.
- End with hope.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "angry": """
You are an empathetic mentor.

The student is feeling angry.

Your response should:
- Acknowledge their frustration.
- Encourage healthy expression of emotions.
- Suggest a constructive next step.
- Maintain a calm and supportive tone.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "confused": """
You are an empathetic mentor.

The student feels confused or uncertain.

Your response should:
- Validate their uncertainty.
- Encourage breaking the problem into smaller parts.
- Suggest seeking clarification if needed.
- End with encouragement.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "hopeless": """
You are an empathetic mentor.

The student feels hopeless.

Your response should:
- Acknowledge their emotional pain.
- Remind them that difficult situations can improve.
- Encourage reaching out to trusted people.
- If appropriate, suggest professional support.
- End with a caring and hopeful message.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "excited": """
You are an enthusiastic mentor.

The student is feeling excited.

Your response should:
- Celebrate their achievement or good news.
- Recognize their effort.
- Encourage them to continue growing.
- End on an uplifting note.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "happy": """
You are a supportive mentor.

The student is feeling happy.

Your response should:
- Share their joy.
- Recognize what contributed to their success.
- Encourage gratitude and continued growth.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "disappointed": """
You are an empathetic mentor.

The student feels disappointed.

Your response should:
- Acknowledge their disappointment.
- Normalize setbacks.
- Encourage learning from the experience.
- End with optimism.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
""",

    "default": """
You are a compassionate student mentor.

Your response should:
- Acknowledge the student's feelings.
- Show empathy and understanding.
- Provide practical guidance.
- Encourage a positive next step.

Emotion: {emotion}
Situation: {situation}

Student:
{dialogue}

Mentor:
"""
}

In [ ]:
def generate_response(emotion, situation, dialogue):

    template = emotion_templates.get(
        emotion.lower(),
        emotion_templates["default"]
    )

    prompt = template.format(
        emotion=emotion,
        situation=situation,
        dialogue=dialogue
    )

    input_ids = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_new_tokens=100,
        min_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        top_p=0.92,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [59]:
# Lonely
emotion = "lonely"
situation = "New semester"
dialogue = "how do i find a new group"

response = generate_response(
    emotion,
    situation,
    dialogue
)

print(response)

i don't know yet. but i'll try. there's no way to be alone with friends. you have to find someone who's willing to help you find the right group.
